<a href="https://colab.research.google.com/github/upeshjeengar/ML-Models-from-scratch/blob/main/LLM%20from%20scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
from torch import nn
import re
from collections import Counter
import math

In [ ]:
!wget https://www.gutenberg.org/files/31447/31447.txt -q
!mv 31447.txt raw_text.txt
with open('raw_text.txt', 'r') as txt:
    content = txt.read()

In [ ]:
result = set(filter(None, re.split(r'[ \n]', content)))
filtered = {s for s in result if not re.search(r'\d', s)}

In [ ]:
vocab = list(filtered)
vocab = sorted(vocab)
stoi = {word: i for i, word in enumerate(vocab)}

raw_words = filter(None, re.split(r'[ \n]', content))
data = [stoi[w] for w in raw_words if w in stoi]

data_tensor = torch.tensor(data, dtype=torch.long)
itos = {i: word for word, i in stoi.items()}

In [ ]:
d_model = 512

embedding_layer = nn.Embedding(num_embeddings=len(vocab), embedding_dim = d_model)

## Encoder Decoder: Attention is all you need

In [ ]:
# --- 2. MODEL CLASSES ---
class PositionalEncoding(nn.Module):
    def __init__(self, d_model=512, max_len=5000, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        seq_len = x.size(1)
        x = x + self.pe[:, :seq_len, :]
        return self.dropout(x)

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model=512, num_heads=8):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def forward(self, query, key, value, mask=None):
        batch_size = query.size(0)

        Q = self.W_q(query).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        K = self.W_k(key).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        V = self.W_v(value).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)

        # FIX: Changed 'attn_scores' to 'scores' to match the variable below
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_model)

        if mask is not None:
            # FIX: Ensure we are checking for 0 correctly (since mask is boolean)
            scores = scores.masked_fill(mask == 0, -1e9)

        attn_weights = torch.softmax(scores, dim=-1)
        context = torch.matmul(attn_weights, V)
        context = context.transpose(1, 2).contiguous().view(batch_size, -1, self.d_model)

        return self.W_o(context)

class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff=2048, dropout=0.1):
        super().__init__()
        self.linear_1 = nn.Linear(d_model, d_ff)
        self.dropout = nn.Dropout(dropout)
        self.linear_2 = nn.Linear(d_ff, d_model)

    def forward(self, x):
        return self.linear_2(self.dropout(torch.relu(self.linear_1(x))))

class EncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.feed_forward = FeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        # FIX: Explicitly pass query, key, value
        attn_output = self.self_attn(query=x, key=x, value=x, mask=mask)
        x = self.norm1(x + self.dropout(attn_output))

        ff_output = self.feed_forward(x)
        x = self.norm2(x + self.dropout(ff_output))
        return x

class DecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.masked_ss_attn = MultiHeadAttention(d_model, num_heads)
        self.enc_dec_attn = MultiHeadAttention(d_model, num_heads)
        self.feed_forward = FeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, enc_output, src_mask, tgt_mask):
        # FIX: Pass query, key, value explicitly
        attn_out = self.masked_ss_attn(query=x, key=x, value=x, mask=tgt_mask)
        x = self.norm1(x + self.dropout(attn_out))

        # FIX: Removed '.forward_explicit' and just call the module directly
        cross_attn_out = self.enc_dec_attn(
            query=x,
            key=enc_output,
            value=enc_output,
            mask=src_mask
        )
        x = self.norm2(x + self.dropout(cross_attn_out))

        ff_out = self.feed_forward(x)
        x = self.norm3(x + self.dropout(ff_out))
        return x

class Transformer(nn.Module):
    def __init__(self, vocab_size, d_model=512, n_layers=6, n_heads=8, d_ff=2048, max_len=5000, dropout=0.1):
        super().__init__()

        # FIX: Added self.d_model so it can be referenced in encode/decode
        self.d_model = d_model

        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoding = PositionalEncoding(d_model, max_len, dropout)

        self.encoder_layers = nn.ModuleList([
            EncoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)
        ])

        self.decoder_layers = nn.ModuleList([
            DecoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)
        ])

        self.fc_out = nn.Linear(d_model, vocab_size)

    def generate_mask(self, src, tgt):
        src_mask = (src != 0).unsqueeze(1).unsqueeze(2)
        tgt_len = tgt.size(1)
        # FIX: Added device=tgt.device so the mask is on the GPU if the data is
        look_ahead_mask = torch.tril(torch.ones((tgt_len, tgt_len), device=tgt.device)).bool()
        return src_mask, look_ahead_mask

    def encode(self, src, src_mask):
        x = self.pos_encoding(self.embedding(src) * math.sqrt(self.d_model))
        for layer in self.encoder_layers:
            x = layer(x, mask=src_mask)
        return x

    def decode(self, tgt, enc_output, src_mask, tgt_mask):
        x = self.pos_encoding(self.embedding(tgt) * math.sqrt(self.d_model))
        for layer in self.decoder_layers:
            x = layer(x, enc_output, src_mask, tgt_mask)
        return x

    def forward(self, src, tgt):
        src_mask, tgt_mask = self.generate_mask(src, tgt)
        enc_output = self.encode(src, src_mask)
        dec_output = self.decode(tgt, enc_output, src_mask, tgt_mask)
        return self.fc_out(dec_output)

# --- 3. TRAINING LOOP ---
def get_batch(data, batch_size, seq_len):
    ix = torch.randint(len(data) - seq_len - 1, (batch_size,))
    src = torch.stack([data[i:i+seq_len] for i in ix])
    tgt = torch.stack([data[i+1:i+seq_len+1] for i in ix])
    return src, tgt

# Device config
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
import torch.optim as optim
from tqdm import tqdm

# Hyperparameters
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = Transformer(vocab_size=len(vocab)).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.0001, betas=(0.9, 0.98), eps=1e-9)
criterion = nn.CrossEntropyLoss()

model.train()

epochs = 1000  # Number of iterations
batch_size = 32
seq_len = 64

for epoch in tqdm(range(epochs)):
    # 1. Get data
    src, tgt = get_batch(data_tensor, batch_size, seq_len)
    src, tgt = src.to(device), tgt.to(device)

    # 2. Forward pass
    # In a real Transformer, 'tgt' is often shifted for the decoder
    # Here we simplify: the model predicts tgt from src
    output = model(src, tgt)

    # 3. Calculate Loss
    # Output shape: (batch, seq_len, vocab_size) -> (batch * seq_len, vocab_size)
    # Tgt shape: (batch, seq_len) -> (batch * seq_len)
    loss = criterion(output.view(-1, len(vocab)), tgt.view(-1))

    # 4. Backpropagation
    optimizer.zero_grad()
    loss.backward()

    # 5. Gradient Clipping (important for Transformers to prevent exploding gradients)
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

    optimizer.step()

    if epoch % 100 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")

  0%|          | 1/1000 [00:18<5:15:14, 18.93s/it]

Epoch 0, Loss: 10.4869


 10%|█         | 101/1000 [28:33<4:08:07, 16.56s/it]

Epoch 100, Loss: 4.3037


 10%|█         | 104/1000 [29:40<4:15:36, 17.12s/it]


KeyboardInterrupt: 

In [ ]:
import torch.nn.functional as F

def diagnostic_generate(model, start_str="this book is ", length=10):
    model.eval()
    words = start_str.split()
    tokens = [stoi[w] for w in words if w in stoi]
    input_tensor = torch.tensor(tokens).unsqueeze(0).to(device)

    print(f"--- Prompt: '{start_str}' ---")

    for step in range(length):
        with torch.no_grad():
            logits = model(input_tensor, input_tensor)
            next_token_logits = logits[:, -1, :]

            # Get probabilities
            probs = F.softmax(next_token_logits, dim=-1)

            # Grab the top 3 probabilities and their indices
            top3_probs, top3_indices = torch.topk(probs, k=3, dim=-1)

            # Print what the model is thinking
            print(f"\nStep {step + 1} Predictions:")
            for i in range(3):
                word = itos[top3_indices[0][i].item()]
                percent = top3_probs[0][i].item() * 100
                print(f"  {i+1}. '{word}' ({percent:.2f}%)")

            # Force it to pick the 1st choice for this test
            next_token = top3_indices[:, 0].unsqueeze(0)
            input_tensor = torch.cat([input_tensor, next_token], dim=1)

    generated_indices = input_tensor.squeeze().tolist()
    print("\n--- Final Output ---")
    print(" ".join([itos[i] for i in generated_indices]))

# Run the diagnostic
diagnostic_generate(model)

--- Prompt: 'this book is ' ---

Step 1 Predictions:
  1. 'is' (94.65%)
  2. 'these' (0.00%)
  3. 'various' (0.00%)

Step 2 Predictions:
  1. 'is' (94.89%)
  2. 'these' (0.00%)
  3. 'this.' (0.00%)

Step 3 Predictions:
  1. 'is' (94.53%)
  2. 'these' (0.00%)
  3. 'this.' (0.00%)

Step 4 Predictions:
  1. 'is' (94.11%)
  2. 'these' (0.00%)
  3. 'this.' (0.00%)

Step 5 Predictions:
  1. 'is' (93.71%)
  2. 'these' (0.00%)
  3. 'era' (0.00%)

Step 6 Predictions:
  1. 'is' (93.37%)
  2. 'these' (0.00%)
  3. 'era' (0.00%)

Step 7 Predictions:
  1. 'is' (93.08%)
  2. 'these' (0.00%)
  3. 'era' (0.00%)

Step 8 Predictions:
  1. 'is' (92.83%)
  2. 'these' (0.00%)
  3. 'era' (0.00%)

Step 9 Predictions:
  1. 'is' (92.61%)
  2. 'these' (0.00%)
  3. 'era' (0.00%)

Step 10 Predictions:
  1. 'is' (92.41%)
  2. 'these' (0.00%)
  3. 'very' (0.00%)

--- Final Output ---
this book is is is is is is is is is is is
Temperature 0.5 (Safe): this book is is is is is is is is is is is is is is is is is is is 

'this book is is is is is is is is is is is is is is is is is is is is is is is is is is is is is is is is is is is is is is is is is is is is is is is is is is is'

## GPT (Decoder Only)

In [ ]:
class GPTBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        # We only need ONE attention mechanism now!
        self.masked_attn = MultiHeadAttention(d_model, num_heads)
        self.feed_forward = FeedForward(d_model, d_ff, dropout)

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        # 1. Masked Self-Attention (Looking back at previous words only)
        attn_out = self.masked_attn(query=x, key=x, value=x, mask=mask)
        x = self.norm1(x + self.dropout(attn_out))

        # 2. Feed Forward
        ff_out = self.feed_forward(x)
        x = self.norm2(x + self.dropout(ff_out))

        return x

class GPTLanguageModel(nn.Module):
    def __init__(self, vocab_size, d_model=512, n_layers=6, n_heads=8, d_ff=2048, max_len=5000, dropout=0.1):
        super().__init__()
        self.d_model = d_model

        # 1. Embeddings & Positional Encoding
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoding = PositionalEncoding(d_model, max_len, dropout)

        # 2. A stack of GPT Blocks (No Encoder!)
        self.blocks = nn.ModuleList([
            GPTBlock(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)
        ])

        # 3. Final Linear Layer to predict the next word
        self.fc_out = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        seq_len = x.size(1)

        # We generate the Look-Ahead mask on the fly right here!
        # This prevents the model from seeing future words.
        mask = torch.tril(torch.ones((seq_len, seq_len), device=x.device)).bool()

        # Apply embeddings + position
        x = self.pos_encoding(self.embedding(x) * math.sqrt(self.d_model))

        # Pass through all blocks
        for block in self.blocks:
            x = block(x, mask=mask)

        # Predict vocab
        return self.fc_out(x)

In [ ]:
# Initialize our new GPT model
model = GPTLanguageModel(vocab_size=len(vocab)).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.0003) # Slightly higher learning rate for GPTs
criterion = nn.CrossEntropyLoss()

model.train()

epochs = 1500  # Let's let it cook a bit longer
batch_size = 32
seq_len = 64

for epoch in range(epochs):
    # 1. Get data
    src, tgt = get_batch(data_tensor, batch_size, seq_len)
    src, tgt = src.to(device), tgt.to(device)

    # 2. Forward pass (Notice we only pass 'src' now!)
    output = model(src)

    # 3. Calculate Loss
    loss = criterion(output.view(-1, len(vocab)), tgt.view(-1))

    # 4. Backpropagation
    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()

    if epoch % 100 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")

Epoch 0, Loss: 10.4780
Epoch 100, Loss: 6.8031
Epoch 200, Loss: 6.5582
